In [1]:
from pyspark.sql import SparkSession

In [2]:
# Iceberg extensions
spark = SparkSession.builder \
    .appName("iceberg-bank-demo") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.5.2") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg.type", "hadoop") \
    .config("spark.sql.catalog.iceberg.warehouse", "iceberg_warehouse") \
    .getOrCreate()

print("Spark with Iceberg is ready")


:: loading settings :: url = jar:file:/home/docker/.local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/docker/.ivy2/cache
The jars for the packages stored in: /home/docker/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.3_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2031ded7-b5a1-4b68-8784-ad4da4b5bce4;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.3_2.12;1.5.2 in central
downloading https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-3.3_2.12/1.5.2/iceberg-spark-runtime-3.3_2.12-1.5.2.jar ...
	[SUCCESSFUL ] org.apache.iceberg#iceberg-spark-runtime-3.3_2.12;1.5.2!iceberg-spark-runtime-3.3_2.12.jar (21944ms)
:: resolution report :: resolve 2042ms :: artifacts dl 21956ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.3_2.12;1.5.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evic

25/11/06 01:35:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark with Iceberg is ready


In [3]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.bank")


DataFrame[]

In [4]:
spark.sql("SHOW NAMESPACES").show(truncate=False)


+---------+
|namespace|
+---------+
|default  |
+---------+



In [5]:
# Create Customers table
spark.sql("""
CREATE TABLE IF NOT EXISTS iceberg.bank.customers (
    customer_id STRING,
    name STRING,
    age INT,
    gender STRING
)
USING iceberg
""")

# Create Accounts table
spark.sql("""
CREATE TABLE IF NOT EXISTS iceberg.bank.accounts (
    account_id STRING,
    customer_id STRING,
    account_type STRING,
    balance DECIMAL(12,2)
)
USING iceberg
""")

# Create Branches table
spark.sql("""
CREATE TABLE IF NOT EXISTS iceberg.bank.branches (
    branch_id STRING,
    branch_name STRING,
    city STRING
)
USING iceberg
""")

# Create Transactions table
spark.sql("""
CREATE TABLE IF NOT EXISTS iceberg.bank.transactions (
    transaction_id STRING,
    account_id STRING,
    branch_id STRING,
    transaction_date DATE,
    transaction_type STRING,
    amount DECIMAL(12,2)
)
USING iceberg
""")

print("All tables created successfully")


All tables created successfully


In [6]:
spark.sql("SHOW TABLES IN iceberg.bank").show(truncate=False)

+---------+------------+-----------+
|namespace|tableName   |isTemporary|
+---------+------------+-----------+
|bank     |transactions|false      |
|bank     |branches    |false      |
|bank     |customers   |false      |
|bank     |accounts    |false      |
+---------+------------+-----------+



In [7]:
spark.sql("""
INSERT INTO iceberg.bank.customers VALUES
('C001', 'Ali Hassan', 35, 'Male'),
('C002', 'Sara Ahmed', 28, 'Female'),
('C003', 'Omar Youssef', 42, 'Male'),
('C004', 'Mona Khaled', 31, 'Female'),
('C005', 'Tarek Nabil', 50, 'Male'),
('C006', 'Layla Mostafa', 24, 'Female')
""")


DataFrame[]

In [8]:
spark.sql("""
INSERT INTO iceberg.bank.accounts VALUES
('A101', 'C001', 'Checking', 15000.50),
('A102', 'C002', 'Savings', 8000.00),
('A103', 'C003', 'Checking', 22000.75),
('A104', 'C004', 'Savings', 12000.00),
('A105', 'C005', 'Checking', 50000.00),
('A106', 'C006', 'Savings', 3000.00)
""")


DataFrame[]

In [9]:
spark.sql("""
INSERT INTO iceberg.bank.branches VALUES
('B001', 'Cairo Main', 'Cairo'),
('B002', 'Alex Center', 'Alexandria'),
('B003', 'Giza West', 'Giza'),
('B004', 'Mansoura Central', 'Mansoura'),
('B005', 'Aswan Branch', 'Aswan')
""")


DataFrame[]

In [10]:
spark.sql("""
INSERT INTO iceberg.bank.transactions VALUES
('T001', 'A101', 'B001', DATE '2025-11-01', 'Deposit', 5000.00),
('T002', 'A102', 'B002', DATE '2025-11-01', 'Withdrawal', 1000.00),
('T003', 'A103', 'B001', DATE '2025-11-02', 'Deposit', 3000.00),
('T004', 'A104', 'B003', DATE '2025-11-03', 'Deposit', 7000.00),
('T005', 'A105', 'B004', DATE '2025-11-03', 'Withdrawal', 2000.00),
('T006', 'A106', 'B002', DATE '2025-11-04', 'Deposit', 1500.00),
('T007', 'A101', 'B001', DATE '2025-11-04', 'Withdrawal', 500.00)
""")


DataFrame[]

# KPIs

In [21]:
#1. Total transaction volume per branch
spark.sql("""
select b.branch_name, sum(t.amount) as total_transaction_volume
from iceberg.bank.transactions t
join iceberg.bank.branches b
on t.branch_id=b.branch_id
group by b.branch_name
order by total_transaction_volume desc""").show()

+----------------+------------------------+
|     branch_name|total_transaction_volume|
+----------------+------------------------+
|      Cairo Main|                 8500.00|
|       Giza West|                 7000.00|
|     Alex Center|                 2500.00|
|Mansoura Central|                 2000.00|
+----------------+------------------------+



In [33]:
#2. Top 5 customers by total deposit amount
spark.sql("""
select c.name , sum(t.amount) as total_deposit_amount
from iceberg.bank.transactions t
join iceberg.bank.accounts a
on t.account_id = a.account_id
join iceberg.bank.customers c
on a.customer_id = c.customer_id
where t.transaction_type ='Deposit'
group by c.name
order by total_deposit_amount desc
""").show(5)

+-------------+--------------------+
|         name|total_deposit_amount|
+-------------+--------------------+
|  Mona Khaled|             7000.00|
|   Ali Hassan|             5000.00|
| Omar Youssef|             3000.00|
|Layla Mostafa|             1500.00|
+-------------+--------------------+



In [27]:
#3. Average transaction amount per account type
spark.sql("""
select a.account_type, round(avg(t.amount), 2) as avg_transaction_amount
from iceberg.bank.transactions t
join iceberg.bank.accounts a 
on t.account_id = a.account_id
group by a.account_type
""").show()


+------------+----------------------+
|account_type|avg_transaction_amount|
+------------+----------------------+
|    Checking|               2625.00|
|     Savings|               3166.67|
+------------+----------------------+



In [26]:
#4. Number of transactions per day
spark.sql("""
select transaction_date, count(*) AS num_transactions
from iceberg.bank.transactions
group by transaction_date
order by transaction_date
""").show()


+----------------+----------------+
|transaction_date|num_transactions|
+----------------+----------------+
|      2025-11-01|               2|
|      2025-11-02|               1|
|      2025-11-03|               2|
|      2025-11-04|               2|
+----------------+----------------+



In [34]:
spark.sql("SELECT * FROM iceberg.bank.transactions.snapshots").show(truncate=False)

+-----------------------+-------------------+---------+---------+-----------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id|operation|manifest_list                                                                                                    |summary                                                                                                                                                                                                                                                                                           |
+-----------------------+-------------------+---------+---

In [35]:
spark.sql("""
INSERT INTO iceberg.bank.transactions VALUES
('T006', 'A003', 'B001', DATE '2025-11-06', 'Deposit', 2000.00)
""")

DataFrame[]

In [36]:
spark.sql("SELECT * FROM iceberg.bank.transactions.snapshots").show(truncate=False)


+-----------------------+-------------------+-------------------+---------+-----------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                    |summary                                                                                                                                                                                                                                                                                           |
+-----------------------+-------------

In [38]:
spark.read.option("snapshot-id", "8947272129877968963").table("iceberg.bank.transactions").show()


+--------------+----------+---------+----------------+----------------+-------+
|transaction_id|account_id|branch_id|transaction_date|transaction_type| amount|
+--------------+----------+---------+----------------+----------------+-------+
|          T001|      A101|     B001|      2025-11-01|         Deposit|5000.00|
|          T002|      A102|     B002|      2025-11-01|      Withdrawal|1000.00|
|          T003|      A103|     B001|      2025-11-02|         Deposit|3000.00|
|          T004|      A104|     B003|      2025-11-03|         Deposit|7000.00|
|          T005|      A105|     B004|      2025-11-03|      Withdrawal|2000.00|
|          T006|      A106|     B002|      2025-11-04|         Deposit|1500.00|
|          T007|      A101|     B001|      2025-11-04|      Withdrawal| 500.00|
+--------------+----------+---------+----------------+----------------+-------+

